In [ ]:
import pandas as pd
import numpy as np
import re

# load admissions / infection-by-age table
# build N_ak = patients by (agegroup, infectionstatus)

df = pd.read_csv("DocumentedInfectionbyAgeGroup.csv")
df_clean = df.iloc[1:].copy()  # row 0 is duplicated header in this file

first_col = df_clean.columns[0]
df_clean[first_col] = (
    df_clean[first_col].astype(str)
      .str.replace("\u2013", "-", regex=False)
      .str.replace("\u2014", "-", regex=False)
      .str.replace(r"\s+", " ", regex=True)
      .str.strip()
)

# standardize labels
df_clean = df_clean[~df_clean[first_col].str.match(r"^total$", case=False, na=False)].copy()
df_clean[first_col] = df_clean[first_col].str.replace(r"^66 and above$", "66+", regex=True)

df_clean.columns = [
    "agegroup",
    "admissions",
    "cai_count", "cai_pct",
    "hai_count", "hai_pct",
    "hbci_count", "hbci_pct",
    "nic_count", "nic_pct",
]

num_cols = ["admissions", "cai_count", "hai_count", "hbci_count", "nic_count"]
for col in num_cols:
    df_clean[col] = pd.to_numeric(df_clean[col], errors="coerce").fillna(0)

coarse_map = {
    "0-29 days": "<1",
    "1 to 11 months": "<1",
    "1 to 5 years": "1-5",
    "6 to 12 years": "6-12",
    "13 to 17 years": "13-17",
    "18 to 29 years": "18-29",
    "30 to 39 years": "30-39",
    "40 to 49 years": "40-49",
    "50 to 65 years": "50-65",
    "66+": "66+",
}

df_clean["agegroup"] = df_clean["agegroup"].map(coarse_map).fillna(df_clean["agegroup"])

# aggregate if multiple fine groups map into same coarse age bin
agg = (
    df_clean.groupby("agegroup", as_index=False)[["admissions", "cai_count", "hai_count", "hbci_count", "nic_count"]]
            .sum()
)

# residual "none" category = admitted but not in one of the four documented infection groups
agg["none_count"] = (
    agg["admissions"]
    - agg["cai_count"]
    - agg["hai_count"]
    - agg["hbci_count"]
    - agg["nic_count"]
).clip(lower=0)

long_rows = []
for _, r in agg.iterrows():
    a = r["agegroup"]
    for k_col, k_name in [
        ("cai_count", "CAI"),
        ("hai_count", "HAI"),
        ("hbci_count", "HBCI"),
        ("nic_count", "NIC"),
        ("none_count", "None"),
    ]:
        long_rows.append({
            "agegroup": a,
            "infectionstatus": k_name,
            "patients": float(r[k_col]),
        })

N_ak = pd.DataFrame(long_rows)

# keep positive-exposure strata only
N_ak = N_ak[N_ak["patients"] > 0].copy()

age_levels = sorted(N_ak["agegroup"].unique().tolist())
inf_levels = sorted(N_ak["infectionstatus"].unique().tolist())

P_patients = float(N_ak["patients"].sum())
N_ak["log_patients"] = np.log(N_ak["patients"])

print("N_ak built.")
print("Total patients across strata:", P_patients)
display(N_ak.head(10))

In [ ]:
# load class table
# get class totals C_i and baseline p0(i|h)

classes = pd.read_csv("AntibioticClassesAcrossHealthFacilities.csv")

def extract_count(x):
    s = str(x)
    m = re.search(r"(\d[\d,]*)", s)
    if not m:
        return 0
    return int(m.group(1).replace(",", ""))

classes["C_i"] = classes["n = 982 (%)"].apply(extract_count)

total_presc = int(classes["C_i"].sum())
assert total_presc == 982, f"Expected total prescriptions = 982, got {total_presc}"

tier_cols = {
    "Specialist": "Specialist (%)",
    "Tertiary":   "Tertiary (%)",
    "District":   "District (%)",
    "Primary":    "Primary (%)",
}

for h, col in tier_cols.items():
    classes[col] = pd.to_numeric(classes[col], errors="coerce").fillna(0.0) / 100.0

rows = []
for _, r in classes.iterrows():
    i = r["Class"]
    Ci = float(r["C_i"])
    for h, col in tier_cols.items():
        rows.append({
            "Class": i,
            "hospital_type": h,
            "count_ih": Ci * float(r[col]),
        })

count_ih = pd.DataFrame(rows)

count_ih["p0_i_given_h"] = (
    count_ih["count_ih"] /
    count_ih.groupby("hospital_type")["count_ih"].transform("sum")
)

C_i = classes.set_index("Class")["C_i"].to_dict()
T_h = count_ih.groupby("hospital_type")["count_ih"].sum().to_dict()

classes_list = classes["Class"].tolist()
H_list = list(tier_cols.keys())

# overall hospital-type shares used to construct synthetic exposures
type_share = {h: T_h[h] / total_presc for h in H_list}

print("Class table loaded.")
print("Total prescriptions:", total_presc)
display(count_ih.head(10))
print("Hospital type shares:", type_share)

In [ ]:
# construct initial target prescription totals by (agegroup, infectionstatus)
# synthetic calibration step before NB-GLM fitting

assert "N_ak" in globals(), "cell 0 must run first"
assert "total_presc" in globals(), "cell 1 must run first"

# provisional intensity multipliers used only to create synthetic targets
# data withheld
age_mult = {a: 1.0 for a in age_levels}
if "<1" in age_mult:
    age_mult["<1"] = 1.15
if "66+" in age_mult:
    age_mult["66+"] = 1.10

# data withheld
inf_mult = {k: 1.0 for k in inf_levels}
for k in inf_mult:
    kk = k.lower()
    if kk == "hai":
        inf_mult[k] = 1.25
    elif kk == "cai":
        inf_mult[k] = 1.10
    elif kk == "hbci":
        inf_mult[k] = 1.05
    elif kk == "nic":
        inf_mult[k] = 0.80
    elif kk == "none":
        inf_mult[k] = 0.70

N_ak = N_ak.copy()
N_ak["raw_m"] = N_ak["agegroup"].map(age_mult) * N_ak["infectionstatus"].map(inf_mult)

raw_total = float((N_ak["patients"] * N_ak["raw_m"]).sum())
scale = total_presc / raw_total
N_ak["m_ak"] = scale * N_ak["raw_m"]

# target prescription totals by (a,k)
N_ak["target_presc_ak"] = N_ak["patients"] * N_ak["m_ak"]

target_presc_ak = (
    N_ak.set_index(["agegroup", "infectionstatus"])["target_presc_ak"].to_dict()
)

# full grid for safety
full_ak = pd.MultiIndex.from_product(
    [age_levels, inf_levels],
    names=["agegroup", "infectionstatus"]
).to_frame(index=False)

N_ak_full = full_ak.merge(
    N_ak[["agegroup", "infectionstatus", "patients", "m_ak", "target_presc_ak"]],
    on=["agegroup", "infectionstatus"],
    how="left"
).fillna({"patients": 0.0, "m_ak": 0.0, "target_presc_ak": 0.0})

print("Initial synthetic target totals built.")
print("Sum target_presc_ak =", N_ak["target_presc_ak"].sum())
display(N_ak_full.head(10))

In [ ]:
# build calibrated synthetic joint table over
# (agegroup, infectionstatus, hospital_type, Class)
# using baseline p0(i|h), severity tilt, and IPF/raking

classes_list = classes["Class"].tolist()
H_list = list(tier_cols.keys())

# data withheld
age_score = {a: 0.0 for a in age_levels}
if "<1" in age_score:
    age_score["<1"] = 0.30
if "66+" in age_score:
    age_score["66+"] = 0.20

inf_score = {k: 0.0 for k in inf_levels}
for k in inf_score:
    kk = k.lower()
    if kk == "hai":
        inf_score[k] = 0.60
    elif kk == "cai":
        inf_score[k] = 0.20
    elif kk == "hbci":
        inf_score[k] = 0.30
    elif kk == "nic":
        inf_score[k] = -0.20
    elif kk == "none":
        inf_score[k] = -0.40

def severity(a, k):
    return age_score.get(a, 0.0) + inf_score.get(k, 0.0)

# class-specific tilt
eta = {i: 0.0 for i in classes_list}
for i in eta:
    name = str(i).lower()
    if ("carbapen" in name) or ("glycopept" in name) or ("vancom" in name):
        eta[i] = 0.8
    elif "penicillin" in name:
        eta[i] = -0.2

p0 = count_ih.set_index(["Class", "hospital_type"])["p0_i_given_h"].to_dict()

# index maps
age_index = {a: idx for idx, a in enumerate(age_levels)}
inf_index = {k: idx for idx, k in enumerate(inf_levels)}
h_index = {h: idx for idx, h in enumerate(H_list)}
i_index = {i: idx for idx, i in enumerate(classes_list)}

A = len(age_levels)
K = len(inf_levels)
H = len(H_list)
I = len(classes_list)

# initial tensor
T = np.zeros((A, K, H, I), dtype=float)

for a in age_levels:
    for k in inf_levels:
        target_ak = float(target_presc_ak.get((a, k), 0.0))
        if target_ak <= 0:
            continue
        sev = severity(a, k)
        weights = []
        keys = []
        for h in H_list:
            for i in classes_list:
                w = type_share[h] * p0.get((i, h), 0.0) * np.exp(eta[i] * sev)
                weights.append(w)
                keys.append((h, i))
        weights = np.asarray(weights, dtype=float)
        if weights.sum() <= 0:
            weights = np.ones_like(weights)
        weights = weights / weights.sum()
        alloc = target_ak * weights
        for (h, i), val in zip(keys, alloc):
            T[age_index[a], inf_index[k], h_index[h], i_index[i]] = val

# ipf targets
target_ak_arr = np.zeros((A, K), dtype=float)
for a in age_levels:
    for k in inf_levels:
        target_ak_arr[age_index[a], inf_index[k]] = float(target_presc_ak.get((a, k), 0.0))

target_hi_arr = np.zeros((H, I), dtype=float)
for h in H_list:
    for i in classes_list:
        target_hi_arr[h_index[h], i_index[i]] = float(
            count_ih.loc[
                (count_ih["hospital_type"] == h) & (count_ih["Class"] == i),
                "count_ih"
            ].sum()
        )

def ipf_4d(T_init, target_ak, target_hi, max_iter=2000, tol=1e-10):
    T_hat = T_init.copy().astype(float)

    for _ in range(max_iter):
        T_prev = T_hat.copy()

        # match (agegroup, infectionstatus) marginals
        current_ak = T_hat.sum(axis=(2, 3))
        with np.errstate(divide="ignore", invalid="ignore"):
            ratio_ak = np.divide(target_ak, current_ak, out=np.ones_like(target_ak), where=current_ak > 0)
        T_hat *= ratio_ak[:, :, None, None]

        # match (hospital_type, Class) marginals
        current_hi = T_hat.sum(axis=(0, 1))
        with np.errstate(divide="ignore", invalid="ignore"):
            ratio_hi = np.divide(target_hi, current_hi, out=np.ones_like(target_hi), where=current_hi > 0)
        T_hat *= ratio_hi[None, None, :, :]

        max_diff = np.max(np.abs(T_hat - T_prev))
        if max_diff < tol:
            break

    return T_hat

T_hat = ipf_4d(T, target_ak_arr, target_hi_arr)

print("Synthetic joint tensor built.")
print("Total synthetic prescriptions =", T_hat.sum())
print("Target total prescriptions   =", total_presc)

In [ ]:
# convert synthetic tensor to dataframe,
# build fitting data,
# fit negative binomial regression,
# estimate alpha and kappa,
# and compute fitted means / variances

import statsmodels.formula.api as smf

# flatten synthetic tensor into a long dataframe
rows = []
for a in age_levels:
    for k in inf_levels:
        for h in H_list:
            for i in classes_list:
                rows.append({
                    "agegroup": a,
                    "infectionstatus": k,
                    "hospital_type": h,
                    "Class": i,
                    "count_float": float(
                        T_hat[
                            age_index[a],
                            inf_index[k],
                            h_index[h],
                            i_index[i]
                        ]
                    )
                })

joint_df = pd.DataFrame(rows)

# compute p(Class | age, infection, hospital_type)
group_sum = joint_df.groupby(
    ["agegroup", "infectionstatus", "hospital_type"]
)["count_float"].transform("sum")

joint_df["p_class"] = 0.0
mask = group_sum > 0
joint_df.loc[mask, "p_class"] = (
    joint_df.loc[mask, "count_float"] / group_sum[mask]
)

# fallback to baseline p0(Class | hospital_type) if a group is degenerate
p0_lookup = count_ih.set_index(["Class", "hospital_type"])["p0_i_given_h"].to_dict()
zero_mask = ~mask
if zero_mask.any():
    joint_df.loc[zero_mask, "p_class"] = joint_df.loc[zero_mask].apply(
        lambda r: p0_lookup.get(
            (r["Class"], r["hospital_type"]),
            1.0 / len(classes_list)
        ),
        axis=1
    )

# construct exposure by (agegroup, infectionstatus, hospital_type)
# exposure_akh = patients(age, infection) * hospital_type share
exposure_df = N_ak[["agegroup", "infectionstatus", "patients"]].copy()
exposure_df["key"] = 1

type_df = pd.DataFrame({
    "hospital_type": H_list,
    "type_share": [type_share[h] for h in H_list],
})
type_df["key"] = 1

stratum_exposure = (
    exposure_df.merge(type_df, on="key", how="outer")
               .drop(columns="key")
)

stratum_exposure["exposure_akh"] = (
    stratum_exposure["patients"] * stratum_exposure["type_share"]
)

joint_df = joint_df.merge(
    stratum_exposure[["agegroup", "infectionstatus", "hospital_type", "exposure_akh"]],
    on=["agegroup", "infectionstatus", "hospital_type"],
    how="left"
)

joint_df["exposure_akh"] = joint_df["exposure_akh"].fillna(0.0)

# build fitting dataframe
# drop negligible synthetic mass
# round to integer counts for NB fit
# keep exposure for offset
fit_df = joint_df.groupby(
    ["agegroup", "infectionstatus", "hospital_type", "Class"],
    as_index=False
).agg(
    count_float=("count_float", "sum"),
    exposure_akh=("exposure_akh", "sum")
)

# drop cells with essentially zero synthetic mass
fit_df = fit_df[fit_df["count_float"] > 1e-6].copy()

# integer counts for NB likelihood
fit_df["count"] = np.round(fit_df["count_float"]).astype(int)

# keep rows that have either positive count or positive exposure
fit_df = fit_df[
    (fit_df["count"] > 0) | (fit_df["exposure_akh"] > 1e-6)
].copy()

fit_df["exposure_akh"] = fit_df["exposure_akh"].clip(lower=1e-8)
fit_df["log_exposure"] = np.log(fit_df["exposure_akh"])

# collapse very rare antimicrobial classes to improve stability
class_totals = fit_df.groupby("Class")["count"].sum().sort_values()
rare_classes = class_totals[class_totals < 10].index.tolist()

fit_df["Class_model"] = fit_df["Class"].where(
    ~fit_df["Class"].isin(rare_classes),
    "Other"
)

print("Rare classes pooled into 'Other':", rare_classes)

# fit NB2 model
# Var(Y|X) = mu + alpha * mu^2
# kappa = 1 / alpha
nb_model = smf.negativebinomial(
    "count ~ C(agegroup) + C(infectionstatus) + C(hospital_type) + C(Class_model)",
    data=fit_df,
    offset=fit_df["log_exposure"]
)

nb_res = nb_model.fit(method="bfgs", maxiter=500, disp=False)

alpha_hat = float(nb_res.params["alpha"])
kappa_hat = np.inf if alpha_hat <= 0 else 1.0 / alpha_hat

# fitted moments on fit_df
fit_df["mu_hat"] = nb_res.predict(fit_df)
fit_df["var_hat"] = fit_df["mu_hat"] + alpha_hat * (fit_df["mu_hat"] ** 2)
fit_df["sd_hat"] = np.sqrt(fit_df["var_hat"])

# robust-radius proxy; can be rescaled later
fit_df["sigma_hat"] = fit_df["sd_hat"]

# map fitted means back to full joint_df
pred_keys = fit_df[[
    "agegroup", "infectionstatus", "hospital_type", "Class", "mu_hat", "var_hat", "sd_hat", "sigma_hat", "Class_model"
]].copy()

joint_df = joint_df.merge(
    pred_keys,
    on=["agegroup", "infectionstatus", "hospital_type", "Class"],
    how="left"
)

# for cells dropped before fitting, fill fitted means with 0
for col in ["mu_hat", "var_hat", "sd_hat", "sigma_hat"]:
    joint_df[col] = joint_df[col].fillna(0.0)

joint_df["Class_model"] = joint_df["Class_model"].fillna(
    joint_df["Class"].where(~joint_df["Class"].isin(rare_classes), "Other")
)

# fitted mean matrix by hospital type and class
mu_hat_hc = (
    joint_df.groupby(["hospital_type", "Class"], as_index=False)["mu_hat"]
            .sum()
)

# display results
print(nb_res.summary())
print(f"\nalpha_hat = {alpha_hat:.6f}")
print(f"kappa_hat = {kappa_hat:.6f}")

print("\nCounts by modeled class:")
display(
    fit_df.groupby("Class_model", as_index=False)["count"]
          .sum()
          .sort_values("count", ascending=False)
)

print("\nPreview of fit_df:")
display(fit_df.head(10))

print("\nPreview of joint_df:")
display(joint_df.head(10))

print("\nPreview of mu_hat_hc:")
display(mu_hat_hc.head(10))

In [ ]:
# export artifacts

from pathlib import Path
import json

out_dir = Path("artifacts")
out_dir.mkdir(exist_ok=True)

# calibrated synthetic joint table
joint_df.to_csv(out_dir / "synthetic_joint_nb_input.csv", index=False)

# conditional probabilities p(Class | age, infection, hospital_type)
pclass_out = joint_df[[
    "agegroup", "infectionstatus", "hospital_type", "Class", "p_class"
]].copy()
pclass_out.to_csv(out_dir / "p_class.csv", index=False)

# target intensity table by (age, infection)
mak_out = N_ak[["agegroup", "infectionstatus", "patients", "m_ak", "target_presc_ak"]].copy()
mak_out.to_csv(out_dir / "m_ak.csv", index=False)

# baseline p0(Class | hospital_type)
p0_out = count_ih[["Class", "hospital_type", "p0_i_given_h"]].copy()
p0_out.to_csv(out_dir / "p0_i_given_h.csv", index=False)

# fitted means by hospital type and class
mu_hat_hc.to_csv(out_dir / "mu_hat_hospital_class.csv", index=False)

# nb parameters
nb_params = pd.DataFrame({
    "parameter": ["alpha_hat", "kappa_hat"],
    "value": [alpha_hat, kappa_hat],
})
nb_params.to_csv(out_dir / "nb_params.csv", index=False)

# full coefficient table
coef_table = pd.DataFrame({
    "term": nb_res.params.index,
    "estimate": nb_res.params.values,
})
coef_table.to_csv(out_dir / "nb_coefficients.csv", index=False)

meta = {
    "classes": classes_list,
    "hospital_types": H_list,
    "age_levels": age_levels,
    "infection_levels": inf_levels,
    "P_patients": float(P_patients),
    "total_prescriptions": int(total_presc),
    "alpha_hat": float(alpha_hat),
    "kappa_hat": None if np.isinf(kappa_hat) else float(kappa_hat),
    "nb2_variance": "Var(Y|X) = mu + alpha * mu^2 = mu + mu^2 / kappa",
}
(out_dir / "metadata.json").write_text(json.dumps(meta, indent=2))

print("Saved artifacts to:", out_dir.resolve())